# 03 · Matching Full Fine-Tune (the showpiece)

Train three ways — **frozen** (floor), **full** fine-tune (reference), and **LoRA**
(two small matrices) — and watch LoRA reach essentially the full-finetune accuracy
while training well under the full parameter count. Then validate the from-scratch
layer against HF `peft` so you trust both.

In [1]:
import sys, pathlib
_p = pathlib.Path.cwd()
while not (_p / 'pyproject.toml').exists() and _p != _p.parent:
    _p = _p.parent
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))
print('repo root:', _p)

repo root: /workspaces/lora_101


In [2]:
import torch
from src.config import get_settings
from src.tasks.toy import make_toy_task, make_base_model
from src.evaluation.comparison import three_way_race
from src.visualisation import plot_race, save_figure
from src.results import save_json

s = get_settings()
task = make_toy_task(dim=s.toy.dim, out_dim=s.toy.out_dim, true_rank=s.toy.true_rank,
                     n_samples=s.toy.n_samples, seed=s.seed)
base = make_base_model(task)
results = three_way_race(base, task, r=s.lora.r, alpha=s.lora.alpha,
                         target_modules=s.lora.target_modules, steps=s.train.steps,
                         lr=s.train.lr, seed=s.seed)
for name in ('frozen', 'full', 'lora'):
    print(results[name])

frozen   loss=1.0007e+00  trainable=0 (0.000%)  0.000 MB  0.00s
full     loss=2.8064e-05  trainable=16,384 (100.000%)  0.062 MB  1.43s
lora     loss=2.9913e-07  trainable=2,048 (11.111%)  0.008 MB  0.50s


## Validate the from-scratch layer against HF `peft`

In [3]:
from src.finetune.peft_ref import (build_peft_single_linear, find_peft_lora_layer,
    peft_adapter_tensors, set_peft_adapter_tensors)
from src.lora.layer import LoRALinear
torch.manual_seed(0)
d = 32
pm = build_peft_single_linear(d, d, s.lora.r, s.lora.alpha, bias=True)
pl = find_peft_lora_layer(pm)
set_peft_adapter_tensors(pl, torch.randn(s.lora.r, d), torch.randn(d, s.lora.r))
scratch = LoRALinear(d, d, r=s.lora.r, alpha=s.lora.alpha, bias=True)
with torch.no_grad():
    scratch.weight.copy_(pl.base_layer.weight)
    scratch.bias.copy_(pl.base_layer.bias)
    a, b = peft_adapter_tensors(pl)
    scratch.lora_A.copy_(a); scratch.lora_B.copy_(b)
xx = torch.randn(5, d)
max_diff = (scratch(xx) - pm(xx)).abs().max().item()
print(f'max|from_scratch - peft| = {max_diff:.2e}  -> {"MATCH" if max_diff < 1e-4 else "DIFF"}')

max|from_scratch - peft| = 0.00e+00  -> MATCH


## Save the race table (JSON) and the figure (PNG)

In [4]:
fig = plot_race(results, title='Frozen vs Full vs LoRA (toy task)')
png = save_figure(fig, '03_three_way_race')
json_path = save_json({
    'race': {name: {'final_loss': rr.final_loss, 'trainable_params': rr.trainable_params,
                    'total_params': rr.total_params, 'trainable_fraction': rr.trainable_fraction,
                    'adapter_mb': rr.adapter_mb, 'seconds': rr.seconds}
             for name, rr in results.items()},
    'peft_validation': {'max_abs_forward_diff': max_diff, 'matches': max_diff < 1e-4},
    'config': {'r': s.lora.r, 'alpha': s.lora.alpha, 'steps': s.train.steps, 'lr': s.train.lr},
}, '03_matching_full_finetune')
print('saved:', png.name, 'and', json_path.name)

saved: 03_three_way_race.png and 03_matching_full_finetune.json


## Takeaway
LoRA sits a hair above the full fine-tune in loss while training a fraction of the
parameters, and it is the *same computation* `peft` performs. Efficiency and parity,
shown together.